# Assignment 3 — IMDB Sentiment & Regularisation

**DATAX504** · Due end of Week 6

Build on **Chapter 5** (regularisation / early stopping) and **Chapter 7** (Keras `compile` / `fit`, callbacks, saving). A compact Sequential stack is fine; you may use the **Functional API** if you prefer (same depth of learning either way).

## Tasks
1. Load IMDB, pad sequences, create a train / validation split
2. Train a **baseline** sentiment model and record best validation accuracy
3. Produce a **diagram** of your model architecture (see §2b)
4. Apply **two** of: L2, dropout, early stopping — report best val accuracy and name the techniques
5. Learning-rate schedule experiment (`ReduceLROnPlateau` or `LearningRateScheduler`)
6. Save the model you consider best with `model.save(...)` and check that it reloads
7. ~200-word reflection on what you learned

## Moodle fields
`a3_repo`, `a3_baseline_val`, `a3_best_val`, `a3_techniques`, `a3_model_file`

## AI disclosure (required)
Fill the table in the next cell before you submit.

## AI disclosure

| Field | Your answer |
|-------|-------------|
| Tool / model | |
| Used for | |
| Verified how | |
| Not used for | |

In [1]:
from os.path import split

import numpy as np
import matplotlib.pyplot as plt
import keras
from keras import layers, regularizers, callbacks

# Optional for plot_model (Ch 7). If import fails, use model.summary() + a drawn sketch (see §2b).
try:
    from keras.utils import plot_model
except ImportError:
    plot_model = None

## 1. Load & prepare IMDB

Hints (fill in the blanks):
- `keras.datasets.imdb.load_data(num_words=...)`
- pad both train and test with `keras.utils.pad_sequences(..., maxlen=...)`
- hold out the **first 10_000** padded training examples as validation (Ch 4 / 5 style split)
- keep the rest of the training set for fitting

In [6]:
max_features = 10000
max_len = 500

(x_train, y_train), (x_test, y_test) = keras.datasets.imdb.load_data(num_words=max_features)

# TODO: pad sequences so every review has length max_len
x_train = keras.utils.pad_sequences(x_train,maxlen = max_len)  # pad_sequences(...)
x_test = keras.utils.pad_sequences(x_test,maxlen = max_len)

# TODO: validation split — first 10_000 samples for val, remainder for train
x_val = x_train[:10000]
y_val = y_train[:10000]
x_train = x_train[10000:]
y_train = y_train[10000:]

print("train:", getattr(x_train, "shape", None), getattr(y_train, "shape", None))
print("val:  ", getattr(x_val, "shape", None), getattr(y_val, "shape", None))
print("test: ", getattr(x_test, "shape", None), getattr(y_test, "shape", None))

train: (15000, 500) (15000,)
val:   (10000, 500) (10000,)
test:  (25000, 500) (25000,)


## 2. Baseline model

Suggested architecture (you may change widths if you document why):
1. `Embedding(max_features, 32, input_length=max_len)` — turns token ids into vectors
2. `GlobalAveragePooling1D()` — mean over the time axis
3. `Dense(32, activation="relu")`
4. `Dense(1, activation="sigmoid")` — binary sentiment

Then:
- `compile` with a suitable optimizer, **binary cross-entropy** loss, and accuracy metric
- `fit` with `validation_data=(x_val, y_val)` for ~10 epochs (batch size is your choice; 512 is common for this data)
- record the **best** validation accuracy over epochs for Moodle `a3_baseline_val`

In [8]:
def build_baseline():
    # TODO: build Sequential (or Functional) model, compile, return it
    model = keras.Sequential([
        layers.Embedding(max_features, 32, input_length=max_len),
        layers.GlobalAveragePooling1D(),
        layers.Dense(32, activation="relu"),
        layers.Dense(1, activation="sigmoid"),
    ])
    model.compile(optimizer="adam",
                  loss="binary_crossentropy",
                  metrics=["accuracy"])
    return model


baseline = build_baseline()

# TODO: fit and keep the History object
hist_baseline = baseline.fit(
    x_train, y_train,
    epochs=10,
    batch_size=512,
    validation_data=(x_val, y_val),
    verbose=1,
 )

# TODO: best validation accuracy across epochs
# baseline_val_acc = max(hist_baseline.history["val_accuracy"])
baseline_val_acc = max(hist_baseline.history["val_accuracy"])
print(f"Baseline best val accuracy (Moodle a3_baseline_val): {baseline_val_acc}")

Epoch 1/10
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.5392 - loss: 0.6914 - val_accuracy: 0.6007 - val_loss: 0.6885
Epoch 2/10
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.5743 - loss: 0.6844 - val_accuracy: 0.6669 - val_loss: 0.6765
Epoch 3/10
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.5978 - loss: 0.6679 - val_accuracy: 0.5683 - val_loss: 0.6673
Epoch 4/10
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.6759 - loss: 0.6351 - val_accuracy: 0.6783 - val_loss: 0.6183
Epoch 5/10
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.7407 - loss: 0.5878 - val_accuracy: 0.7176 - val_loss: 0.5729
Epoch 6/10
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.7785 - loss: 0.5334 - val_accuracy: 0.8042 - val_loss: 0.5130
Epoch 7/10
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.8160 - loss: 0.4772 - val_accuracy: 0.8080 - val_loss: 0.4692
Epoch 8/10
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.8369 - loss: 0.4306 - val_accuracy: 0.8243 - v

## 2b. Diagram of your architecture (required)

You *can* do this with the tools from **Chapter 7 / Week 5**:

```python
keras.utils.plot_model(
    baseline,                      # or your regularised model
    to_file="imdb_model.png",      # commit this image to your repo
    show_shapes=True,
    show_layer_names=True,
)
```

Also print `model.summary()` so layer shapes appear in the notebook.

**If `plot_model` fails** (missing Graphviz / pydot on your machine):
1. Keep a full `model.summary()` output in the notebook, and
2. Add a short markdown diagram or hand-drawn photo of the stack (Embedding → … → sigmoid).

Either way, a reader should see the full forward path without reading all of your code.

In [ ]:
# TODO: print architecture summary
baseline.summary()

# TODO: save a DAG figure when Graphviz/pydot are available
DIAGRAM_FILE = "imdb_model.png"
if plot_model is not None:
    try:
        plot_model(
            baseline,  # swap for reg_model later if you prefer that diagram
            to_file=DIAGRAM_FILE,
            show_shapes=True,
            show_layer_names=True,
        )
        print(f"Wrote {DIAGRAM_FILE} — commit it to your GitHub repo")
        # Optional: display in notebook
        # from IPython.display import Image, display
        # display(Image(DIAGRAM_FILE))
    except Exception as e:
        print("plot_model failed — use summary + hand diagram:", e)
else:
    print("plot_model unavailable — use model.summary() and a markdown/hand diagram")

## 3. Regularised model

Choose **exactly two** (or more if curious, but Moodle asks for two) of:

| Technique | Typical hook |
|-----------|----------------|
| L2 | `kernel_regularizer=regularizers.l2(...)` on a Dense layer |
| Dropout | `layers.Dropout(rate)` after a Dense / before the output |
| Early stopping | `callbacks.EarlyStopping(monitor="val_loss", patience=..., restore_best_weights=True)` |

Hints:
- Set boolean flags below so you can print which techniques you used
- `compile` again after building the stack
- Pass a **list of callbacks** into `fit` (empty list if you did not pick early stopping)
- Train long enough that early stopping might fire (e.g. more epochs than the baseline)
- Best val accuracy → Moodle `a3_best_val`; names → `a3_techniques`

In [ ]:
USE_L2 = False          # set True if using L2
USE_DROPOUT = False     # set True if using dropout
USE_EARLY_STOP = False  # set True if using EarlyStopping

# Example L2 strength if USE_L2:
# l2_reg = regularizers.l2(1e-4)

reg_model = keras.Sequential([
    # TODO: Embedding + pooling as in baseline
    # TODO: Dense with optional kernel_regularizer=
])
# if USE_DROPOUT:
#     reg_model.add(layers.Dropout(...))
# reg_model.add(layers.Dense(1, activation="sigmoid"))

# TODO: compile

cb = []
# if USE_EARLY_STOP:
#     cb.append(callbacks.EarlyStopping(...))  # restore_best_weights=True is recommended (Week 5)

# TODO: fit → hist_reg
# hist_reg = reg_model.fit(..., callbacks=cb, ...)

best_val_acc = None  # max(hist_reg.history["val_accuracy"])
techniques = []
if USE_L2:
    techniques.append("L2")
if USE_DROPOUT:
    techniques.append("dropout")
if USE_EARLY_STOP:
    techniques.append("early stopping")

print(f"Techniques (Moodle a3_techniques): {', '.join(techniques)}")
print(f"Best val accuracy (Moodle a3_best_val): {best_val_acc}")
assert len(techniques) >= 2, "Enable (at least) two techniques for full credit"

## 4. Learning-rate schedule

Week 5 / Ch 7 callback: try **`ReduceLROnPlateau`** (shrink LR when `val_loss` stalls)
or a custom **`LearningRateScheduler`**.

Hints:
- Start from the baseline or your regularised architecture (`build_baseline()` is fine)
- Put the schedule callback in the `callbacks=[...]` list for `fit`
- After training, plot the learning-rate history if present (`hist.history.get("lr", [])` for ReduceLROnPlateau)
- Short markdown below: which schedule, why those hyperparameters, what you observed

In [ ]:
lr_model = build_baseline()  # or rebuild your regularised architecture

# TODO: choose one schedule callback, e.g.
# lr_cb = callbacks.ReduceLROnPlateau(
#     monitor="val_loss", factor=0.5, patience=1, min_lr=1e-6, verbose=1
# )

# hist_lr = lr_model.fit(
#     x_train, y_train,
#     epochs=...,
#     batch_size=...,
#     validation_data=(x_val, y_val),
#     callbacks=[lr_cb],  # may combine with EarlyStopping
#     verbose=1,
# )

# TODO: plot LR vs epoch when the history key exists
# lrs = hist_lr.history.get("lr", [])
# plt.plot(lrs)
# plt.xlabel("Epoch")
# plt.ylabel("Learning rate")
# plt.title("LR schedule")
# plt.show()

**LR experiment notes** (a few sentences):

- Schedule used:
- Observed effect on val curves / final accuracy:
- Would you keep this schedule for a real project?

## 5. Save best model

Hints (Ch 7):
- Pick one of baseline / regularised / LR-tuned
- `best_model.save("something.keras")` — full architecture + weights
- `keras.models.load_model(...)` and `evaluate` on **test** once (do not retune on test)
- Commit the `.keras` file and the diagram PNG to your GitHub repo
- Filename → Moodle `a3_model_file`

In [ ]:
MODEL_FILE = "imdb_sentiment_best.keras"  # Moodle a3_model_file

# TODO: choose the model object you consider best
# best_model = reg_model  # or baseline or lr_model

# TODO: save
# best_model.save(MODEL_FILE)

# TODO: reload and evaluate on the test set once
# loaded = keras.models.load_model(MODEL_FILE)
# _, test_acc = loaded.evaluate(x_test, y_test, verbose=0)

print(f"Saved model filename (Moodle a3_model_file): {MODEL_FILE}")
# print(f"Test accuracy of saved model: {test_acc:.4f}")

## 6. What did you learn?

Write ~200 words in the cell below (or paste the same text into Moodle if asked).
Cover: baseline vs regularised result, whether a technique hurt or helped, LR schedule takeaway, and one thing you would try next.

*Your reflection (~200 words).*